# Notebook 03 — ML Modelling
**Student 3 · Modelling Lead**

Implements walk-forward cross-sectional ML to predict 21-day forward returns.
Models: Naive Baseline, Ridge Regression, Random Forest, XGBoost.
Outputs `results/predictions.csv` and `results/model_metrics.csv`.

**Prerequisites:** Run `02_features.ipynb` first.

In [ ]:
# ── Cell 1 · Setup & Imports ──────────────────────────────────────────────────
import os, sys, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('WARNING: xgboost not installed. XGB model will be skipped.')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

PROC_DIR    = '../data/processed'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete.')

In [ ]:
# ── Cell 2 · Load Features & Walk-Forward Setup ───────────────────────────────
features_df = pd.read_csv(os.path.join(PROC_DIR, 'features.csv'), parse_dates=['date'])
features_df.sort_values(['date', 'ticker'], inplace=True)
features_df.reset_index(drop=True, inplace=True)

FEATURE_COLS = [c for c in features_df.columns
                if c.startswith('f_')]
TARGET_COL   = 'target_21d'

print(f'Features shape : {features_df.shape}')
print(f'Feature columns: {FEATURE_COLS}')
print(f'Date range     : {features_df.date.min().date()} → {features_df.date.max().date()}')

# ── Walk-forward parameters ───────────────────────────────────────────────────
TRAIN_START  = '2015-01-01'
TRAIN_END    = '2019-12-31'   # initial training window end
TEST_START   = '2020-01-01'
PURGE_DAYS   = 21             # purge between train end and test start
RETRAIN_FREQ = 'YE'           # retrain yearly

# Generate monthly prediction dates (last trading day of each month in test period)
all_dates     = features_df['date'].unique()
all_dates_s   = pd.DatetimeIndex(sorted(all_dates))
test_dates    = all_dates_s[all_dates_s >= TEST_START]
month_ends    = test_dates.to_series().groupby(
    test_dates.to_series().dt.to_period('M')).last().values

print(f'\nMonthly prediction dates: {len(month_ends)}')
print(f'First: {pd.Timestamp(month_ends[0]).date()}  Last: {pd.Timestamp(month_ends[-1]).date()}')

In [ ]:
# ── Cell 3 · Naive Baseline ───────────────────────────────────────────────────
# Historical mean return per stock computed on training data

def naive_predict(train_df, test_df):
    """Predict each ticker's historical mean target from training data."""
    means = train_df.groupby('ticker')[TARGET_COL].mean()
    preds = test_df['ticker'].map(means)
    return preds.values

print('Naive baseline model defined.')
# Will be called inside walk-forward loop below.

In [ ]:
# ── Cell 4 · Ridge Regression ─────────────────────────────────────────────────
def ridge_predict(X_train, y_train, X_test):
    """Ridge with cross-validated alpha inside the training window."""
    alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
    model  = RidgeCV(alphas=alphas, cv=5)
    model.fit(X_train, y_train)
    return model.predict(X_test), model

print('Ridge model defined.')

In [ ]:
# ── Cell 5 · Random Forest ────────────────────────────────────────────────────
def rf_predict(X_train, y_train, X_test):
    """Shallow Random Forest (max_depth=3)."""
    model = RandomForestRegressor(
        n_estimators=200, max_depth=3,
        min_samples_leaf=20, random_state=RANDOM_SEED,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model.predict(X_test), model

print('Random Forest model defined.')

In [ ]:
# ── Cell 6 · XGBoost ──────────────────────────────────────────────────────────
def xgb_predict(X_train, y_train, X_test):
    """XGBoost with max_depth=3 and learning_rate=0.05."""
    model = xgb.XGBRegressor(
        max_depth=3, learning_rate=0.05,
        n_estimators=300, subsample=0.8,
        colsample_bytree=0.8, random_state=RANDOM_SEED,
        verbosity=0, n_jobs=-1
    )
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test_dummy := np.zeros(len(X_test)))],
              verbose=False)
    return model.predict(X_test), model

if not XGB_AVAILABLE:
    def xgb_predict(X_train, y_train, X_test):
        return np.zeros(len(X_test)), None

print('XGBoost model defined.')

In [ ]:
# ── Cell 7 · Walk-Forward Prediction Loop ─────────────────────────────────────
MODELS = ['naive', 'ridge', 'rf', 'xgb']
all_predictions = []

# Clean rows: drop where any feature or target is NaN
clean_df = features_df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

print(f'Clean rows for modelling: {len(clean_df)}')

# Cache trained models per retrain year
trained_models   = {m: None for m in MODELS}
trained_scalers  = {m: None for m in MODELS}
last_train_year  = None
last_train_df_   = None

for pred_date in month_ends:
    pred_date = pd.Timestamp(pred_date)
    year      = pred_date.year

    # Determine training window (expanding)
    train_cutoff = pred_date - pd.Timedelta(days=PURGE_DAYS)

    if year != last_train_year:
        # Retrain on all data up to train_cutoff
        train_df = clean_df[clean_df['date'] <= train_cutoff]
        if len(train_df) < 100:
            continue  # not enough data yet

        X_train = train_df[FEATURE_COLS].values
        y_train = train_df[TARGET_COL].values

        # Fit scaler on train data
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)

        # Train models
        _, ridge_model = ridge_predict(X_train_sc, y_train, X_train_sc[:1])
        _, rf_model    = rf_predict(X_train_sc, y_train, X_train_sc[:1])
        if XGB_AVAILABLE:
            _, xgb_model = xgb_predict(X_train_sc, y_train, X_train_sc[:1])
        else:
            xgb_model = None

        # Naive: compute per-ticker means
        naive_means = train_df.groupby('ticker')[TARGET_COL].mean().to_dict()

        trained_models  = {'ridge': ridge_model, 'rf': rf_model, 'xgb': xgb_model}
        trained_scalers = scaler
        last_train_year  = year
        last_train_df_   = train_df
        print(f'  Retrained @ {pred_date.date()}  train_rows={len(train_df)}')

    # Test slice: stocks available on prediction date
    test_df = features_df[features_df['date'] == pred_date].copy()
    if test_df.empty:
        # Find nearest date
        near = clean_df[clean_df['date'] <= pred_date]['date'].max()
        if pd.isna(near):
            continue
        test_df = features_df[features_df['date'] == near].copy()

    test_df_clean = test_df.dropna(subset=FEATURE_COLS)
    if test_df_clean.empty:
        continue

    X_test     = test_df_clean[FEATURE_COLS].values
    X_test_sc  = trained_scalers.transform(X_test)
    actual     = test_df_clean[TARGET_COL].values

    for model_name in MODELS:
        if model_name == 'naive':
            preds = test_df_clean['ticker'].map(naive_means).values
        elif model_name == 'ridge':
            preds = trained_models['ridge'].predict(X_test_sc)
        elif model_name == 'rf':
            preds = trained_models['rf'].predict(X_test_sc)
        elif model_name == 'xgb':
            if trained_models['xgb'] is not None:
                preds = trained_models['xgb'].predict(X_test_sc)
            else:
                preds = np.zeros(len(X_test_sc))

        for i, (_, row) in enumerate(test_df_clean.iterrows()):
            all_predictions.append({
                'date': pred_date,
                'ticker': row['ticker'],
                'model': model_name,
                'pred': float(preds[i]) if not np.isnan(preds[i]) else 0.0,
                'actual': float(actual[i]) if not np.isnan(actual[i]) else np.nan
            })

predictions_df = pd.DataFrame(all_predictions)
print(f'\nTotal predictions: {len(predictions_df)}')
print(predictions_df.head())

In [ ]:
# ── Cell 8 · Evaluation Metrics ───────────────────────────────────────────────
def compute_ic(pred_series, actual_series):
    """Rank IC (Spearman) between predictions and actuals."""
    mask = ~(np.isnan(pred_series) | np.isnan(actual_series))
    if mask.sum() < 5:
        return np.nan
    r, _ = stats.spearmanr(pred_series[mask], actual_series[mask])
    return r

def top_minus_bottom(pred_series, actual_series, q=5):
    """Mean actual return of top-q stocks minus bottom-q."""
    df = pd.DataFrame({'pred': pred_series, 'actual': actual_series}).dropna()
    if len(df) < 2 * q:
        return np.nan
    df_s = df.sort_values('pred', ascending=False)
    return df_s.iloc[:q]['actual'].mean() - df_s.iloc[-q:]['actual'].mean()

metrics_rows = []
for model_name in MODELS:
    model_preds = predictions_df[predictions_df['model'] == model_name].copy()

    # Monthly IC
    monthly_ic = model_preds.groupby('date').apply(
        lambda g: compute_ic(g['pred'].values, g['actual'].values)
    ).dropna()

    ic_mean  = monthly_ic.mean()
    ic_std   = monthly_ic.std()
    ic_tstat = ic_mean / (ic_std / np.sqrt(len(monthly_ic))) if ic_std > 0 else np.nan

    # RMSE
    mp = model_preds.dropna(subset=['actual'])
    rmse = np.sqrt(mean_squared_error(mp['actual'], mp['pred']))

    # Directional accuracy
    dir_acc = (np.sign(mp['pred']) == np.sign(mp['actual'])).mean()

    # Top-Bottom spread
    tmb = model_preds.groupby('date').apply(
        lambda g: top_minus_bottom(g['pred'].values, g['actual'].values)
    ).dropna().mean()

    metrics_rows.append({
        'model': model_name, 'period': 'full',
        'ic_mean': round(ic_mean, 4),
        'ic_tstat': round(ic_tstat, 3) if not np.isnan(ic_tstat) else np.nan,
        'rmse': round(rmse, 6),
        'dir_acc': round(dir_acc, 4),
        'top_minus_bottom': round(tmb, 4) if not np.isnan(tmb) else np.nan
    })
    print(f'{model_name:8s}  IC={ic_mean:.4f}  t={ic_tstat:.2f}  RMSE={rmse:.5f}  DirAcc={dir_acc:.3f}  TMB={tmb:.4f}')

model_metrics_df = pd.DataFrame(metrics_rows)
print('\n=== Model Metrics ===')
print(model_metrics_df.to_string(index=False))

In [ ]:
# ── Cell 9 · Feature Importance Charts ───────────────────────────────────────
# Re-train final RF and XGB on full data for importance plots
clean_all = features_df.dropna(subset=FEATURE_COLS + [TARGET_COL])
X_all = clean_all[FEATURE_COLS].values
y_all = clean_all[TARGET_COL].values
scaler_all = StandardScaler()
X_all_sc = scaler_all.fit_transform(X_all)

rf_final = RandomForestRegressor(
    n_estimators=200, max_depth=3, min_samples_leaf=20,
    random_state=RANDOM_SEED, n_jobs=-1)
rf_final.fit(X_all_sc, y_all)

fig, axes = plt.subplots(1, 2 if XGB_AVAILABLE else 1, figsize=(14, 5))
if not XGB_AVAILABLE:
    axes = [axes]

# RF importance
rf_imp = pd.Series(rf_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
rf_imp.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Random Forest Feature Importance')
axes[0].set_xlabel('Importance')

if XGB_AVAILABLE:
    xgb_final = xgb.XGBRegressor(
        max_depth=3, learning_rate=0.05, n_estimators=300,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, verbosity=0, n_jobs=-1)
    xgb_final.fit(X_all_sc, y_all)
    xgb_imp = pd.Series(xgb_final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    xgb_imp.plot.barh(ax=axes[1], color='darkorange')
    axes[1].set_title('XGBoost Feature Importance')
    axes[1].set_xlabel('Importance')

plt.suptitle('Feature Importances (trained on full dataset)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importances.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/feature_importances.png')

In [ ]:
# ── Cell 10 · Save Outputs ────────────────────────────────────────────────────
pred_path    = os.path.join(RESULTS_DIR, 'predictions.csv')
metrics_path = os.path.join(RESULTS_DIR, 'model_metrics.csv')

predictions_df.to_csv(pred_path, index=False)
model_metrics_df.to_csv(metrics_path, index=False)

print(f'Saved predictions.csv    {predictions_df.shape}')
print(f'Saved model_metrics.csv  {model_metrics_df.shape}')
print('\n✅  Notebook 03 complete.')